# Safe to Be Challenged — Pilot Notebook

Trust and harmlessness in adversarial AI advising. This notebook runs the three-condition experiment (committee-only, silent monitor, visible label) using digital doubles from the MACSS corpus, then summarizes results and optionally fetches GitHub memos for longitudinal validation.

In [ ]:
# Colab: install deps with pip (use -e . if you uploaded the project)
!pip install -q httpx beautifulsoup4 sentence-transformers transformers accelerate bitsandbytes numpy scikit-learn matplotlib seaborn

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from safe_to_be_challenged.config import CONDITIONS
from safe_to_be_challenged.data import CorpusLoader, DoublesLoader, GitHubIssuesLoader
from safe_to_be_challenged.metrics import mechanical_reliance, quality_rating, trust_rating
from safe_to_be_challenged.models import CommitteeLoader, EmbeddingLoader, SafetyLoader

## Load models and data

Load committee, embeddings, safety monitor; then corpus and digital doubles.

In [ ]:
embed_loader = EmbeddingLoader()
committee = CommitteeLoader()
safety = SafetyLoader(committee)

corpus = CorpusLoader()
doubles_loader = DoublesLoader(corpus)
doubles = doubles_loader.load_doubles_from_corpus(n_per_condition=2)
print(f"Loaded {len(doubles)} doubles across conditions: {CONDITIONS}")

## Define feedback, revision, and baseline

Wrappers used in the experiment loop.

In [ ]:
def get_feedback(d):
    q = committee.skill_diagnostician(d["thesis"])
    critique = committee.adversarial_critic(d["thesis"], q)
    if d["condition"] == "committee_only":
        return critique
    return safety.harmlessness_monitor(critique, d["condition"], generate_fn=committee.generate)

def pure_llm_baseline(thesis):
    return committee.generate(f"Revise this thesis abstract to improve it. {thesis[:400]}", max_new_tokens=200)

def student_revision(thesis, feedback, d):
    if d["trust_sensitivity"] == "low" and d["condition"] != "visible_label":
        return pure_llm_baseline(thesis)
    prompt = f"Revise this thesis incorporating the feedback. Thesis: {thesis[:350]}\nFeedback: {feedback[:400]}"
    return committee.generate(prompt, max_new_tokens=200)

## Run experiment loop

For each double: get feedback, revision, baseline; compute embeddings and metrics.

In [ ]:
results = []
for d in doubles:
    feedback = get_feedback(d)
    rev = student_revision(d["thesis"], feedback, d)
    base = pure_llm_baseline(d["thesis"])

    emb_rev = embed_loader.get_embeddings([rev])[0]
    emb_base = embed_loader.get_embeddings([base])[0]
    mr = mechanical_reliance(emb_rev, emb_base, embed_loader.cosine_sim)
    trust = trust_rating(rev, committee.generate)
    quality = quality_rating(rev, committee.generate)

    results.append({
        "condition": d["condition"],
        "mechanical_reliance": float(mr),
        "trust": trust,
        "quality": quality,
    })
print(f"Completed {len(results)} runs.")

## Summary table and plots

In [ ]:
# Table: mean by condition
for cond in CONDITIONS:
    sub = [r for r in results if r["condition"] == cond]
    if sub:
        mr_mean = np.mean([r["mechanical_reliance"] for r in sub])
        trust_mean = np.mean([r["trust"] for r in sub])
        qual_mean = np.mean([r["quality"] for r in sub])
        print(f"{cond}: MR={mr_mean:.3f}, trust={trust_mean:.2f}, quality={qual_mean:.2f} (n={len(sub)})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, cond in enumerate(CONDITIONS):
    sub = [r for r in results if r["condition"] == cond]
    if sub:
        axes[0].bar(i, np.mean([r["mechanical_reliance"] for r in sub]), label=cond, alpha=0.8)
        axes[1].bar(i, np.mean([r["trust"] for r in sub]), label=cond, alpha=0.8)
axes[0].set_xticks(range(len(CONDITIONS)))
axes[0].set_xticklabels([c.replace("_", "\n") for c in CONDITIONS])
axes[0].set_ylabel("Mean mechanical reliance")
axes[0].set_title("Mechanical reliance (lower = less copy-paste)")
axes[1].set_xticks(range(len(CONDITIONS)))
axes[1].set_xticklabels([c.replace("_", "\n") for c in CONDITIONS])
axes[1].set_ylabel("Mean trust (engagement)")
axes[1].set_title("Trust by condition")
plt.tight_layout()
plt.show()

## Optional: fetch GitHub issues for longitudinal validation

Uncomment and run to fetch Week N Memo comments from the course repo.

In [ ]:
# gh = GitHubIssuesLoader()
# gh_corpus = gh.load_github_corpus(force_refresh=True)
# print(f"Fetched {len(gh_corpus)} memo comments")
# gh.close()

---

Pilot complete. See `report.md` and `README.md` for method and qualitative interpretation.